####Задача 1:

Посмотрим поближе на кодирование и декодирование символов.

In [ ]:
s1 = 'привет'
b = s.encode('utf-8')
print(b, len(b))
s2 = b.decode('utf-8')
print(s2, len(s2))

b'\xd0\xbf\xd1\x80\xd0\xb8\xd0\xb2\xd0\xb5\xd1\x82' 12
привет 6


Cимволы и байты не одно и то же! Различаем типы данных:

- str — текст, последовательность символов Unicode.

- bytes — сырые байты, то, что реально лежит на диске.

Мост между ними — кодировка. Только она помогает нам превратить непонятный набор чисел в читаемый текст.

Как вы думаете, почему для кодирования шести букв потребовалось целых 12 байт?

Подсказка в следующем коде.

In [ ]:
char1 = 'д'
print(len(char1.encode('utf-8')))
char2 = 'd'
print(len(char2.encode('utf-8')))

2
1


####Задача 2:

Теперь посмотрим на то, какие бывают кодировки.

Допустим, у нас есть очень важный текстовый файл, но мы не знаем, в какой кодировке он записан. Что делать?

Если мы попробуем прочитать его в неправильной кодировке, получим ошибку.

In [ ]:
from os import chdir as cd

cd('..')
with open(r'usr/f.txt', 'r', encoding='utf-8') as f:
    print(f.read())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf0 in position 0: invalid continuation byte

Чтобы программа точно не упала, используем параметр errors='replace' или errors='ignore' команды open.

In [ ]:
from os import chdir as cd

cd('..')
with open(r'usr/f.txt', 'r', encoding='utf-8', errors='replace') as f:  #'replace' заменит все нечитаемые символы на �, 'ignore' просто уберет их.
    print(f.read())

������, ��� ��� ���������. ��� ����� ������!


Теперь мы можем открыть файл и даже узнать, что в нем есть какой-то "живой" текст, но все равно не можем прочитать этот текст, если кодировка оказалась не 'utf-8'. Нам нужно проверить другие.

Пробуем по очереди, пока не прочитается без ошибок.

In [3]:
from os import chdir as cd

cd('..')
for enc in ('utf-8', 'utf-16be', 'cp1251', 'koi8-r'):
    with open(r'usr/f.txt', 'r', encoding=enc, errors='ignore') as f: #на этот раз используем 'ignore', чтобы посмотреть на чистый результат без замен
        print(enc, ":", f.read())

utf-8 : ,   .   !
utf-16be : 짗엔Ⱐ켠췏씠폏쿂컉씮⃯컏⃏컘⃗쇖컏씡
cp1251 : рТЙЧЕФ, ЬФП НПЕ УППВЭЕОЙЕ. пОП ПЮЕОШ ЧБЦОПЕ!
koi8-r : Привет, это мое сообщение. Оно очень важное!


Ура, мы прочитали!

На помощь нам пришла кодировка KOI8-R (Код Обмена Информацией, 8-битный, Русский) — одна из самых знаковых и исторически важных кириллических кодировок.


**Историческая справка:** Разработка семейства кодировок KOI8 началась еще в 1970-х годах в СССР для советских компьютеров и мейнфреймов (серии ЕС ЭВМ, СМ ЭВМ).Сам стандарт KOI8-R в его современном виде был формализован в 1993 году в документе RFC 1489. В 1990-х годах она стала де-факто главным стандартом для зарождающегося рунета. На ней держался весь ранний российский интернет: веб-страницы, первые форумы и, самое главное, электронная почта в операционных системах Unix и Linux.


Теперь посмотреть на этот же файл по-другому. Нас ждет кое-что интересное!

In [ ]:
from os import chdir as cd

cd('..')
# 1. Читаем наш файл как сырые байты
with open(r'usr/f.txt', "rb") as f:
    koi8_bytes = f.read()

# 2. Очищаем старший бит у каждого байта (магия 90-х: делаем & 127 или % 128)
ascii_bytes = bytes([b & 0x7F for b in koi8_bytes])

# 3. Декодируем получившийся результат в обычную латиницу кодировкой ASCII
ascii_text = ascii_bytes.decode("ascii")

print(ascii_text)

pRIWET, \TO MOE SOOB]ENIE. oNO O^ENX WAVNOE!


Мы получили с трудом, но читаемый транслит в кодировке ASCII (англ. American Standard Code for Information Interchange, современный ее вариант называется ANSII).

Буквы в «перевернутом» регистре (заглавные вместо строчных и наоборот), но смысл сообщения доходит до пользователя, даже если у него нет таблиц кириллических кодировок. Это основная фишка кодировки KOI8-R, которая реально спасала людей в 90-х.

Если у 8-битного символа KOI8-R отбросить старший, восьмой бит - например, если письмо проходило через почтовый сервер или старый терминал умел работать только с 7-битным английским текстом, - то текст не превращается в хаотичный набор знаков, русские буквы переходят в соответствующие им по звучанию латинские буквы (правда, не все, но большинство).

Попробуем еще одну старую кодировку.

In [ ]:
from os import chdir as cd

cd('..')
with open(r'usr/f.txt', 'r', encoding='koi8-u', errors='replace') as f:
    print(f.read())

Привет, это мое сообщение. Оно очень важное!


Это KOI8-U, другой вариант KOI8. Он был утвержден в 1998 году для украинского языка. В кодировку KOI8-R добавили 4 буквы украинского алфавита: Ґ, Є, І, Ї (и их строчные варианты). Чтобы освободить место для украинских букв в KOI8-U, создателям пришлось пожертвовать некоторыми символами псевдографики (рамками, линиями), которые были в KOI8-R. При этом все русские буквы остались на тех же самых местах, поэтому KOI8-U обратно совместима с KOI8-R для русского текста.

**Примечание:** Сегодня KOI8 практически полностью вытеснена UTF-8, но помнить о старой кодировке нужно не только по историческим причинам, но и потому, что столкнуться с файлами в ней все еще можно в сферах

- Legacy-систем в банках и госсекторе: Старые базы данных, написанные под Unix/Linux в 90-х годах, которые работают десятилетиями без модернизации.

- Сетевого и промышленного оборудования: Внутренние логи, старые АТС, системы мониторинга или станки с ЧПУ.

- Архивов электронных писем (Email): Старые почтовые ящики и бэкапы писем из 90-х и начала 2000-х годов.

- Авиации (протокол SITATEX): В международной авиационной текстовой связи до сих пор используются старые стандарты передачи данных, где иногда проскакивают реликты KOI8-R.